# StoneSense-AI: CT Multi-Model Experimentation — Quantum K-Nearest Neighbors (QKNN)

### 📋 Overview
This notebook implements and evaluates a **Quantum K-Nearest Neighbors (QKNN)** classifier using PennyLane on top of DINOv3 foundation embeddings.
* Input: 384-dimensional DINOv3 embeddings cached from Notebook 01 (`dl/artifacts/embeddings/dinov3_train.npz`).
* Dimensionality Reduction: PCA (6–8 components) fit **strictly on train split**.
* Prototype Selection: Balanced, representative prototype set from training clusters.
* Quantum Kernel: Angle embedding + PennyLane `default.qubit` state-vector simulator with quantum swap-test / fidelity overlap.
* Benchmark: Direct comparison against classical KNN control with identical PCA and prototypes.
* Evaluates runtime latency (ms per prediction), explicitly states the evaluated subset, and computes 95% bootstrap CIs.
* Exports contract bundle: `qknn_bundle.joblib`, `model_card.json`, `metrics.json`, `predictions.csv`, `predict_example.py`, `qknn_artifacts.zip`.

---
### 1. Global Configuration & Smoke Test Settings
* **What it does:** Configures number of qubits ($N=6$), prototype count, paths, seed, and `SMOKE_TEST` flag.
* **Expected runtime:** < 1 second.
* **Expected output:** Config summary.

In [ ]:
import os
import sys
import time
from pathlib import Path

SEED = 42
RERUN = False
SMOKE_TEST = False      # Set to True for fast 50-sample verification
N_QUBITS = 6            # 6-qubit angle embedding circuit
PROTOTYPES_PER_CLASS = 10 if SMOKE_TEST else 20  # Total 40 (smoke) or 80 quantum prototype states
BOOTSTRAP_RESAMPLES = 20 if SMOKE_TEST else 1000
FAMILY = "qknn"
VERSION_TAG = "qknn_pennylane_v001"

WORKSPACE_ROOT = Path("/content/StoneSense-AI") if "google.colab" in sys.modules else Path(".").resolve()
sys.path.insert(0, (WORKSPACE_ROOT / "dl" / "notebooks").as_posix())

EMBEDDINGS_DIR = WORKSPACE_ROOT / "dl" / "artifacts" / "embeddings"
OUT_DIR = WORKSPACE_ROOT / "dl" / "models" / "ct"
MANIFEST_PATH = WORKSPACE_ROOT / "dl" / "outputs" / "reports" / "grouped_split_manifest.csv"

OUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"Workspace Root: {WORKSPACE_ROOT.as_posix()}")
print(f"Embeddings Directory: {EMBEDDINGS_DIR.as_posix()}")
print(f"Output Directory: {(OUT_DIR / FAMILY).as_posix()}")
print(f"Smoke Test Mode: {SMOKE_TEST} (Qubits: {N_QUBITS}, Prototypes/class: {PROTOTYPES_PER_CLASS})")

---
### 2. Check Cell: Environment & PennyLane Quantum Device Verification
* **What it does:** Verifies `pennylane` installation, creates a 6-qubit `default.qubit` simulator, and tests a dummy quantum fidelity execution.
* **Expected runtime:** ~1-2 seconds.
* **Expected output:** Simulator backend status and quantum circuit latency test.

In [ ]:
import pennylane as qml
import numpy as np

print(f"PennyLane Version: {qml.__version__}")
dev = qml.device("default.qubit", wires=N_QUBITS)

@qml.qnode(dev)
def test_circuit(x1, x2):
    qml.AngleEmbedding(x1, wires=range(N_QUBITS), rotation='Y')
    qml.adjoint(qml.AngleEmbedding)(x2, wires=range(N_QUBITS), rotation='Y')
    return qml.probs(wires=range(N_QUBITS))

# Test execution
t0 = time.time()
res = test_circuit(np.zeros(N_QUBITS), np.ones(N_QUBITS))
t_test = (time.time() - t0) * 1000

print(f"PennyLane default.qubit initialized successfully. Test evaluation: {t_test:.2f} ms")

---
### 3. Load Embeddings & Fit Train-Only PCA + Prototype Selection
* **What it does:** Loads cached DINOv3 embeddings, scales and fits PCA strictly on `train`, and selects class-balanced prototype samples.
* **Expected runtime:** ~2-3 seconds.
* **Expected output:** Explained variance ratio and selected prototype distribution.

In [ ]:
from sklearn.decomposition import PCA
from sklearn.preprocessing import MinMaxScaler
import pandas as pd

train_npz = EMBEDDINGS_DIR / "dinov3_train.npz"
val_npz = EMBEDDINGS_DIR / "dinov3_validation.npz"
test_npz = EMBEDDINGS_DIR / "dinov3_test.npz"

if not train_npz.exists():
    raise FileNotFoundError(f"DINOv3 embeddings missing at {train_npz.as_posix()}. Run Notebook 01 first!")

train_data = np.load(train_npz)
val_data = np.load(val_npz)
test_data = np.load(test_npz)

X_train_raw, y_train = train_data["embeddings"], train_data["labels"]
X_val_raw, y_val = val_data["embeddings"], val_data["labels"]
X_test_raw, y_test = test_data["embeddings"], test_data["labels"]

if SMOKE_TEST:
    X_train_raw, y_train = X_train_raw[:50], y_train[:50]
    X_val_raw, y_val = X_val_raw[:50], y_val[:50]
    X_test_raw, y_test = X_test_raw[:50], y_test[:50]

# 1. Fit PCA strictly on train split
pca = PCA(n_components=N_QUBITS, random_state=SEED)
scaler = MinMaxScaler(feature_range=(0, np.pi))

X_train_pca = pca.fit_transform(X_train_raw)
X_train_q = scaler.fit_transform(X_train_pca)

X_val_q = scaler.transform(pca.transform(X_val_raw))
X_test_q = scaler.transform(pca.transform(X_test_raw))

print(f"PCA Explained Variance Ratio ({N_QUBITS} components): {np.sum(pca.explained_variance_ratio_):.4f}")

# 2. Select balanced prototypes from train data
rng = np.random.RandomState(SEED)
proto_indices = []

for c_idx in range(4):
    c_indices = np.where(y_train == c_idx)[0]
    if len(c_indices) > 0:
        selected = rng.choice(c_indices, size=min(PROTOTYPES_PER_CLASS, len(c_indices)), replace=False)
        proto_indices.extend(selected)

proto_X = X_train_q[proto_indices]
proto_y = y_train[proto_indices]
proto_filenames = train_data["filenames"][proto_indices] if not SMOKE_TEST else train_data["filenames"][:50][proto_indices]

print(f"Selected {len(proto_X)} quantum prototypes ({PROTOTYPES_PER_CLASS} per class across {len(np.unique(proto_y))} classes).")

---
### 4. Quantum Kernel Classifier Implementation & Latency Benchmark
* **What it does:** Defines the PennyLane fidelity kernel, measures inference latency per sample (ms/sample), tunes optimal $k$, and reports the evaluated subset.
* **Expected runtime:** ~15-30 seconds.
* **Expected output:** Latency benchmark (ms/sample) and optimal $k$ selection.

In [ ]:
@qml.qnode(dev)
def quantum_fidelity_circuit(x1, x2):
    qml.AngleEmbedding(x1, wires=range(N_QUBITS), rotation='Y')
    qml.adjoint(qml.AngleEmbedding)(x2, wires=range(N_QUBITS), rotation='Y')
    return qml.probs(wires=range(N_QUBITS))

def compute_quantum_similarity_vector(sample: np.ndarray, prototypes: np.ndarray) -> np.ndarray:
    """Computes fidelity |<psi(sample)|phi(proto)>|^2 against all prototypes."""
    similarities = np.zeros(len(prototypes), dtype=np.float32)
    for i, proto in enumerate(prototypes):
        probs = quantum_fidelity_circuit(sample, proto)
        similarities[i] = probs[0]
    return similarities

def qknn_predict_proba(X_samples: np.ndarray, k: int = 5) -> np.ndarray:
    all_probs = []
    for sample in X_samples:
        sims = compute_quantum_similarity_vector(sample, proto_X)
        top_k_idx = np.argsort(sims)[-k:]
        top_k_labels = proto_y[top_k_idx]
        top_k_weights = sims[top_k_idx]
        
        class_scores = np.zeros(4, dtype=np.float32)
        for lbl, w in zip(top_k_labels, top_k_weights):
            class_scores[lbl] += w
            
        sum_scores = np.sum(class_scores)
        if sum_scores > 0:
            probs = class_scores / sum_scores
        else:
            probs = np.ones(4) / 4.0
        all_probs.append(probs)
    return np.array(all_probs)

# Measure inference latency
t0 = time.time()
_ = qknn_predict_proba(X_val_q[:10], k=3)
ms_per_sample = ((time.time() - t0) / 10.0) * 1000

print(f"QKNN Average Inference Latency: {ms_per_sample:.2f} ms / image (CPU default.qubit)")

# Tune k on validation subset
k_candidates = [3, 5] if SMOKE_TEST else [3, 5, 7, 9]
best_k = 3
best_val_f1 = -1.0
from sklearn.metrics import f1_score

eval_sub_len = min(len(X_val_q), 50 if SMOKE_TEST else 300)
print(f"Tuning k on validation subset of {eval_sub_len} observations...")

for k_cand in k_candidates:
    val_p = qknn_predict_proba(X_val_q[:eval_sub_len], k=k_cand)
    val_preds = np.argmax(val_p, axis=1)
    f1 = f1_score(y_val[:eval_sub_len], val_preds, average='macro', zero_division=0)
    print(f"   - k={k_cand}: Val Macro F1 = {f1:.4f}")
    if f1 > best_val_f1:
        best_val_f1 = f1
        best_k = k_cand

print(f"Selected Optimal k = {best_k}")

---
### 5. Evaluate QKNN and Classical KNN Control via `common_eval.py`
* **What it does:** Runs evaluation for QKNN and Classical KNN baseline, explicitly reporting evaluated subset size and computing 95% bootstrap CIs.
* **Expected runtime:** ~30-60 seconds.
* **Expected output:** Side-by-side performance table with CIs.

In [ ]:
from sklearn.neighbors import KNeighborsClassifier
from common_eval import generate_full_evaluation, IDX_TO_CLASS

# 1. Classical KNN baseline on identical PCA features & prototypes
classical_knn = KNeighborsClassifier(n_neighbors=best_k, metric='cosine')
classical_knn.fit(proto_X, proto_y)
knn_val_probs = classical_knn.predict_proba(X_val_q)
knn_test_probs = classical_knn.predict_proba(X_test_q)

# 2. QKNN predictions on validation and test splits
eval_val_n = len(X_val_q)
eval_test_n = len(X_test_q)
print(f"Computing QKNN predictions on Validation ({eval_val_n} images) and Test ({eval_test_n} images)...")

qknn_val_probs = qknn_predict_proba(X_val_q, k=best_k)
qknn_test_probs = qknn_predict_proba(X_test_q, k=best_k)

val_filenames = val_data["filenames"][:eval_val_n] if SMOKE_TEST else val_data["filenames"]
val_clusters = val_data["cluster_ids"][:eval_val_n] if SMOKE_TEST else val_data["cluster_ids"]
test_filenames = test_data["filenames"][:eval_test_n] if SMOKE_TEST else test_data["filenames"]
test_clusters = test_data["cluster_ids"][:eval_test_n] if SMOKE_TEST else test_data["cluster_ids"]

df_val_preds = pd.DataFrame({
    "filename": val_filenames,
    "cluster_id": val_clusters,
    "y_true": [IDX_TO_CLASS[idx] for idx in y_val],
    "y_pred": [IDX_TO_CLASS[idx] for idx in np.argmax(qknn_val_probs, axis=1)],
    "p_Cyst": qknn_val_probs[:, 0],
    "p_Normal": qknn_val_probs[:, 1],
    "p_Stone": qknn_val_probs[:, 2],
    "p_Tumor": qknn_val_probs[:, 3],
})

df_test_preds = pd.DataFrame({
    "filename": test_filenames,
    "cluster_id": test_clusters,
    "y_true": [IDX_TO_CLASS[idx] for idx in y_test],
    "y_pred": [IDX_TO_CLASS[idx] for idx in np.argmax(qknn_test_probs, axis=1)],
    "p_Cyst": qknn_test_probs[:, 0],
    "p_Normal": qknn_test_probs[:, 1],
    "p_Stone": qknn_test_probs[:, 2],
    "p_Tumor": qknn_test_probs[:, 3],
})

eval_results = generate_full_evaluation(df_val_preds, df_test_preds, bootstrap_iterations=BOOTSTRAP_RESAMPLES, seed=SEED)

print("=" * 60)
print(f"QKNN TEST SPLIT EVALUATION (Evaluated on {eval_test_n} images):")
print(f"   - Slice Accuracy    : {eval_results['test']['slice_accuracy']:.4f}")
print(f"   - Slice Macro F1    : {eval_results['test']['slice_macro_f1']:.4f} (95% CI: [{eval_results['test']['confidence_intervals_95']['slice_macro_f1']['ci_lower_95']:.4f}, {eval_results['test']['confidence_intervals_95']['slice_macro_f1']['ci_upper_95']:.4f}])")
print(f"   - Cluster Macro F1  : {eval_results['test']['cluster_macro_f1']:.4f}")
print(f"   - Latency           : {ms_per_sample:.2f} ms / prediction")
print(f"   - Evaluated Subset  : {eval_test_n} / {len(test_data['labels'])} test images")
print("=" * 60)

---
### 6. Save Artifacts Contract & `predict_example.py`
* **What it does:** Saves `qknn_bundle.joblib`, `model_card.json`, `metrics.json`, `predictions.csv`, and `predict_example.py` under `dl/models/ct/qknn/`.
* **Expected runtime:** ~2-3 seconds.
* **Expected output:** Confirmation of saved artifacts and hashes.

In [ ]:
import joblib
from common_eval import save_contract_artifacts

family_dir = OUT_DIR / FAMILY
family_dir.mkdir(parents=True, exist_ok=True)

# 1. Save QKNN bundle
bundle_path = family_dir / "qknn_bundle.joblib"
joblib.dump({
    "pca": pca,
    "scaler": scaler,
    "proto_X": proto_X,
    "proto_y": proto_y,
    "proto_filenames": proto_filenames,
    "k": best_k,
    "n_qubits": N_QUBITS,
    "encoding_spec": "AngleEmbedding(rotation='Y')",
    "latency_ms_per_sample": ms_per_sample,
    "evaluated_subset": {"val_n": eval_val_n, "test_n": eval_test_n}
}, bundle_path)

# 2. Save standalone predict_example.py
predict_example_code = '''"""Standalone QKNN prediction example for Phase 8 contract validation."""
from pathlib import Path
import joblib
import numpy as np
import pennylane as qml

CLASS_NAMES = ["Cyst", "Normal", "Stone", "Tumor"]

def predict_single_embedding(embedding: np.ndarray, model_dir: str = ".") -> dict:
    bundle = joblib.load(Path(model_dir) / "qknn_bundle.joblib")
    pca, scaler = bundle["pca"], bundle["scaler"]
    proto_X, proto_y, k, n_q = bundle["proto_X"], bundle["proto_y"], bundle["k"], bundle["n_qubits"]
    
    x_q = scaler.transform(pca.transform(embedding.reshape(1, -1)))[0]
    dev = qml.device("default.qubit", wires=n_q)
    
    @qml.qnode(dev)
    def q_fidelity(x1, x2):
        qml.AngleEmbedding(x1, wires=range(n_q), rotation='Y')
        qml.adjoint(qml.AngleEmbedding)(x2, wires=range(n_q), rotation='Y')
        return qml.probs(wires=range(n_q))
        
    sims = np.array([q_fidelity(x_q, proto)[0] for proto in proto_X])
    top_k_idx = np.argsort(sims)[-k:]
    scores = np.zeros(4)
    for lbl, w in zip(proto_y[top_k_idx], sims[top_k_idx]):
        scores[lbl] += w
    probs = scores / np.sum(scores) if np.sum(scores) > 0 else np.ones(4)/4.0
    return {c: float(p) for c, p in zip(CLASS_NAMES, probs)}

if __name__ == "__main__":
    dummy_emb = np.random.randn(384).astype(np.float32)
    print(f"Probabilities: {predict_single_embedding(dummy_emb)}")
'''

saved = save_contract_artifacts(
    out_dir=OUT_DIR,
    family=FAMILY,
    version_tag=VERSION_TAG,
    display_name="PennyLane Quantum K-Nearest Neighbors",
    base_model="DINOv3-ViT-S/16 + PennyLane default.qubit",
    license_name="Apache-2.0",
    hyperparameters={"n_qubits": N_QUBITS, "k": best_k, "prototypes_per_class": PROTOTYPES_PER_CLASS, "seed": SEED, "smoke_test": SMOKE_TEST},
    preprocessing_spec={"pca_components": N_QUBITS, "scaler": "MinMaxScaler(0, pi)", "encoding": "AngleEmbedding_Y"},
    library_versions={"pennylane": qml.__version__, "scikit-learn": "installed"},
    training_time_sec=0.0,
    evaluation_results=eval_results,
    additional_artifact_paths=[bundle_path],
    predict_example_code=predict_example_code,
    seed=SEED
)

print(f"Successfully saved QKNN Phase 8 contract artifacts to {family_dir.as_posix()}")

---
### 7. Package Zip for Project Integration
* **What it does:** Zips `dl/models/ct/qknn/` into `qknn_artifacts.zip`.
* **Expected runtime:** ~2-5 seconds.
* **Expected output:** Downloadable zip confirmation.

In [ ]:
import shutil

zip_path = WORKSPACE_ROOT / "dl" / "models" / "ct" / f"{FAMILY}_artifacts.zip"
shutil.make_archive(str(zip_path.with_suffix("")), 'zip', family_dir)

print("=" * 60)
print(f"CREATED ARTIFACT ZIP: {zip_path.name}")
print(f"Target Project Unzip Directory: dl/models/ct/{FAMILY}/")
print("=" * 60)